# 🏦 Advanced Loan Approval System — LendingClub Dataset

**Pipeline Overview:**
1. Data Loading & Initial EDA
2. Data Cleaning & Feature Engineering
3. Model Training: XGBoost, LightGBM, Logistic Regression (Stacked Ensemble)
4. Threshold Tuning (Business-Aware: minimise bad loan approvals)
5. SHAP Explainability
6. Model Export for Web App
7. Dashboard Metrics Export


In [ ]:
# ── 0. INSTALL DEPENDENCIES ──────────────────────────────────────────────────
!pip install -q kaggle lightgbm shap imbalanced-learn optuna joblib plotly

In [ ]:
# ── 1. KAGGLE SETUP & DATA DOWNLOAD ─────────────────────────────────────────
# Option A: Upload kaggle.json via Colab files panel, then run this cell
# Option B: Mount Google Drive if dataset is already there

import os
from google.colab import files

print('Upload your kaggle.json API key:')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d wordsforthewise/lending-club --unzip -p /content/data/
!ls /content/data/

In [ ]:
# ── 2. IMPORTS ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
import json
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, StackingClassifier, RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_recall_curve, roc_curve, f1_score, average_precision_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV

import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
import optuna
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 100)
RANDOM_STATE = 42
print('All imports successful ✅')

In [ ]:
# ── 3. LOAD DATA ─────────────────────────────────────────────────────────────
# LendingClub accepted loans file
df = pd.read_csv('/content/data/accepted_2007_to_2018Q4.csv/accepted_2007_to_2018Q4.csv',
                 low_memory=False)

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()[:30]}...')
df.head(3)

In [ ]:
# ── 4. TARGET VARIABLE CREATION ──────────────────────────────────────────────
# loan_status → binary: 1 = default/charged-off (BAD), 0 = fully paid (GOOD)
print('Loan Status Distribution:')
print(df['loan_status'].value_counts())

BAD_STATUSES = [
    'Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off',
    'Late (31-120 days)', 'Late (16-30 days)'
]
GOOD_STATUSES = [
    'Fully Paid', 'Does not meet the credit policy. Status:Fully Paid'
]

# Keep only loans that have resolved (drop 'Current', 'In Grace Period' etc.)
df = df[df['loan_status'].isin(BAD_STATUSES + GOOD_STATUSES)].copy()
df['target'] = df['loan_status'].isin(BAD_STATUSES).astype(int)

print(f'\nFiltered Shape: {df.shape}')
print(f"Class distribution:\n{df['target'].value_counts(normalize=True).round(3)}")

In [ ]:
# ── 5. FEATURE SELECTION (Domain-Driven) ─────────────────────────────────────
# Select features available AT TIME OF APPLICATION (no data leakage)
APPLICATION_FEATURES = [
    # Loan characteristics
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
    'purpose', 'title',
    # Borrower financials
    'annual_inc', 'dti', 'emp_length', 'home_ownership',
    # Credit history
    'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec', 'revol_bal',
    'revol_util', 'total_acc', 'earliest_cr_line', 'inq_last_6mths',
    'mths_since_last_delinq', 'delinq_2yrs', 'pub_rec_bankruptcies',
    'mort_acc', 'num_bc_sats', 'pct_tl_nvr_dlq',
    # Address
    'addr_state', 'zip_code',
    # Target
    'target'
]

# Only keep columns that exist
available = [c for c in APPLICATION_FEATURES if c in df.columns]
df = df[available].copy()
print(f'Features kept: {len(available) - 1} (+ target)')
print(df.isnull().sum().sort_values(ascending=False).head(15))

In [ ]:
# ── 6. DATA CLEANING ─────────────────────────────────────────────────────────

def clean_lending_club(df):
    df = df.copy()

    # --- term: '36 months' → 36
    if 'term' in df.columns:
        df['term'] = df['term'].str.extract(r'(\d+)').astype(float)

    # --- int_rate: '15.02%' → 15.02
    if 'int_rate' in df.columns:
        df['int_rate'] = df['int_rate'].astype(str).str.replace('%','').astype(float)

    # --- revol_util: '54.6%' → 54.6
    if 'revol_util' in df.columns:
        df['revol_util'] = df['revol_util'].astype(str).str.replace('%','').astype(float)

    # --- emp_length: '10+ years' → 10, '< 1 year' → 0
    if 'emp_length' in df.columns:
        df['emp_length'] = df['emp_length'].str.extract(r'(\d+)').astype(float)
        df['emp_length'].fillna(df['emp_length'].median(), inplace=True)

    # --- earliest_cr_line → credit history length in years
    if 'earliest_cr_line' in df.columns:
        df['issue_year'] = 2018  # approximate
        df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
        df['credit_history_yrs'] = (2018 - df['earliest_cr_line'].dt.year).clip(0, 50)
        df.drop(columns=['earliest_cr_line', 'issue_year'], inplace=True)

    # --- fico: average of range
    if 'fico_range_low' in df.columns and 'fico_range_high' in df.columns:
        df['fico_score'] = (df['fico_range_low'] + df['fico_range_high']) / 2
        df.drop(columns=['fico_range_low', 'fico_range_high'], inplace=True)

    # --- zip_code: keep only first 3 digits
    if 'zip_code' in df.columns:
        df['zip3'] = df['zip_code'].str[:3]
        df.drop(columns=['zip_code'], inplace=True)

    # --- drop title (free text, high cardinality)
    if 'title' in df.columns:
        df.drop(columns=['title'], inplace=True)

    # --- mths_since_last_delinq: NaN → never delinquent → encode as high value
    if 'mths_since_last_delinq' in df.columns:
        df['mths_since_last_delinq'].fillna(999, inplace=True)

    return df

df = clean_lending_club(df)
print('Cleaning done. Shape:', df.shape)

In [ ]:
# ── 7. FEATURE ENGINEERING ───────────────────────────────────────────────────

def engineer_features(df):
    df = df.copy()

    # Debt-to-income risk bucket
    df['dti_bucket'] = pd.cut(df['dti'], bins=[-1, 10, 20, 30, 40, 100],
                               labels=[0,1,2,3,4]).astype(float)

    # Loan-to-income ratio
    df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'].clip(lower=1))

    # Monthly payment burden
    if 'installment' in df.columns and 'annual_inc' in df.columns:
        df['payment_to_income'] = (df['installment'] * 12) / (df['annual_inc'].clip(lower=1))

    # FICO band
    if 'fico_score' in df.columns:
        df['fico_band'] = pd.cut(df['fico_score'],
                                  bins=[0,580,670,740,800,900],
                                  labels=['Poor','Fair','Good','Very Good','Exceptional']).astype(str)

    # Credit utilisation flag
    if 'revol_util' in df.columns:
        df['high_util_flag'] = (df['revol_util'] > 75).astype(int)

    # Clean derogatory marks
    if 'pub_rec' in df.columns:
        df['has_pub_rec'] = (df['pub_rec'] > 0).astype(int)
    if 'pub_rec_bankruptcies' in df.columns:
        df['has_bankruptcy'] = (df['pub_rec_bankruptcies'] > 0).astype(int)

    # Interest rate tier
    if 'int_rate' in df.columns:
        df['int_rate_tier'] = pd.cut(df['int_rate'], bins=[0,7,12,18,25,40],
                                      labels=['Very Low','Low','Mid','High','Very High']).astype(str)

    return df

df = engineer_features(df)
print('Feature engineering done. Shape:', df.shape)

In [ ]:
# ── 8. ENCODE CATEGORICALS ───────────────────────────────────────────────────
CAT_COLS = df.select_dtypes(include=['object']).columns.tolist()
print('Categorical columns:', CAT_COLS)

# Drop high cardinality columns
HIGH_CARD = ['addr_state', 'zip3']  # keep but target-encode

# Label encode
le_dict = {}
for col in CAT_COLS:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le

print('\nFinal dtypes:')
print(df.dtypes.value_counts())

In [ ]:
# ── 9. TRAIN/TEST SPLIT ──────────────────────────────────────────────────────
# Drop any remaining NaN
df.dropna(subset=['target'], inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

FEATURE_COLS = [c for c in df.columns if c != 'target']
X = df[FEATURE_COLS]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Sub-sample for speed in Colab (remove if you have full GPU time)
MAX_TRAIN = 400_000
if len(X_train) > MAX_TRAIN:
    X_train = X_train.sample(MAX_TRAIN, random_state=RANDOM_STATE)
    y_train = y_train.loc[X_train.index]

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Class balance (train): {y_train.mean():.3f} default rate')

In [ ]:
# ── 10. HANDLE CLASS IMBALANCE (SMOTE) ──────────────────────────────────────
# Only apply SMOTE to training set
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'After SMOTE — Train: {X_train_res.shape}, Balance: {y_train_res.mean():.3f}')

In [ ]:
# ── 11. HYPERPARAMETER TUNING WITH OPTUNA ────────────────────────────────────
# Tune XGBoost (5-trial fast run — increase n_trials for production)

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 600),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'use_label_encoder': False,
        'eval_metric': 'auc',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'tree_method': 'hist',  # GPU: change to 'gpu_hist'
    }
    model = xgb.XGBClassifier(**params)
    score = cross_val_score(model, X_train_res, y_train_res, cv=3,
                             scoring='roc_auc', n_jobs=-1).mean()
    return score

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')
study.optimize(objective_xgb, n_trials=15, show_progress_bar=True)

best_xgb_params = study.best_params
best_xgb_params.update({'use_label_encoder': False, 'eval_metric': 'auc',
                          'random_state': RANDOM_STATE, 'n_jobs': -1, 'tree_method': 'hist'})
print(f'Best XGB AUC: {study.best_value:.4f}')
print('Best params:', best_xgb_params)

In [ ]:
# ── 12. TRAIN INDIVIDUAL MODELS ──────────────────────────────────────────────

# --- XGBoost (tuned)
xgb_model = xgb.XGBClassifier(**best_xgb_params)
xgb_model.fit(X_train_res, y_train_res,
               eval_set=[(X_test, y_test)],
               verbose=False)
xgb_proba = xgb_model.predict_proba(X_test)[:,1]
print(f'XGBoost AUC: {roc_auc_score(y_test, xgb_proba):.4f}')

# --- LightGBM
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=RANDOM_STATE, n_jobs=-1
)
lgb_model.fit(X_train_res, y_train_res,
               eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50, verbose=False)])
lgb_proba = lgb_model.predict_proba(X_test)[:,1]
print(f'LightGBM AUC: {roc_auc_score(y_test, lgb_proba):.4f}')

# --- Logistic Regression (calibrated)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_test_sc  = scaler.transform(X_test)

lr_model = LogisticRegression(C=0.1, max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
lr_model.fit(X_train_sc, y_train_res)
lr_proba = lr_model.predict_proba(X_test_sc)[:,1]
print(f'Logistic Regression AUC: {roc_auc_score(y_test, lr_proba):.4f}')

In [ ]:
# ── 13. STACKING ENSEMBLE ────────────────────────────────────────────────────
from sklearn.ensemble import StackingClassifier

estimators = [
    ('xgb', xgb.XGBClassifier(**best_xgb_params)),
    ('lgb', lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)),
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(C=1.0, max_iter=1000),
    cv=3,
    stack_method='predict_proba',
    n_jobs=-1
)

stack_model.fit(X_train_res, y_train_res)
stack_proba = stack_model.predict_proba(X_test)[:,1]
stack_auc = roc_auc_score(y_test, stack_proba)
print(f'Stacking Ensemble AUC: {stack_auc:.4f}')

In [ ]:
# ── 14. BUSINESS-AWARE THRESHOLD TUNING ──────────────────────────────────────
# Cost matrix: approving a bad loan is more expensive than rejecting a good one
# Assume: FN (missed default) cost = $5000, FP (rejected good loan) cost = $200

COST_FN = 5000  # missed default (approve bad loan)
COST_FP = 200   # rejected good loan (opportunity cost)

thresholds = np.arange(0.1, 0.9, 0.01)
costs = []

for t in thresholds:
    preds = (stack_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    total_cost = (fn * COST_FN) + (fp * COST_FP)
    costs.append(total_cost)

optimal_threshold = thresholds[np.argmin(costs)]
print(f'Optimal business threshold: {optimal_threshold:.2f}')

plt.figure(figsize=(10, 4))
plt.plot(thresholds, costs)
plt.axvline(optimal_threshold, color='red', linestyle='--',
             label=f'Optimal: {optimal_threshold:.2f}')
plt.xlabel('Decision Threshold')
plt.ylabel('Total Business Cost ($)')
plt.title('Business Cost vs Decision Threshold')
plt.legend()
plt.tight_layout()
plt.savefig('/content/threshold_cost.png', dpi=150)
plt.show()

In [ ]:
# ── 15. FINAL EVALUATION ─────────────────────────────────────────────────────
final_preds = (stack_proba >= optimal_threshold).astype(int)

print('='*60)
print('FINAL MODEL EVALUATION (Stacking Ensemble)')
print('='*60)
print(f'ROC-AUC:  {roc_auc_score(y_test, stack_proba):.4f}')
print(f'PR-AUC:   {average_precision_score(y_test, stack_proba):.4f}')
print(f'F1:       {f1_score(y_test, final_preds):.4f}')
print(f'Threshold: {optimal_threshold:.2f}')
print()
print(classification_report(y_test, final_preds, target_names=['Good Loan','Bad Loan']))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, stack_proba)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr, tpr, label=f'Stacking AUC={stack_auc:.3f}')
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend()

# Confusion Matrix
cm = confusion_matrix(y_test, final_preds)
sns.heatmap(cm, annot=True, fmt='d', ax=axes[1],
            xticklabels=['Predicted Good','Predicted Bad'],
            yticklabels=['Actual Good','Actual Bad'])
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig('/content/model_evaluation.png', dpi=150)
plt.show()

In [ ]:
# ── 16. SHAP EXPLAINABILITY ──────────────────────────────────────────────────
# Use XGBoost for SHAP (fastest)
explainer = shap.TreeExplainer(xgb_model)

# Use a sample for speed
X_shap = X_test.sample(min(2000, len(X_test)), random_state=RANDOM_STATE)
shap_values = explainer.shap_values(X_shap)

# Global Feature Importance
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig('/content/shap_importance.png', dpi=150)
plt.show()

# SHAP Beeswarm
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_shap, show=False)
plt.tight_layout()
plt.savefig('/content/shap_beeswarm.png', dpi=150)
plt.show()

In [ ]:
# ── 17. INDIVIDUAL PREDICTION EXPLAINER ──────────────────────────────────────
# Explain a single applicant decision
sample_idx = 0
sample = X_test.iloc[[sample_idx]]
sample_shap = explainer.shap_values(sample)
prob = xgb_model.predict_proba(sample)[0][1]

print(f'Applicant default probability: {prob:.2%}')
print(f'Decision: {"REJECT" if prob >= optimal_threshold else "APPROVE"}')

shap.waterfall_plot(
    shap.Explanation(values=sample_shap[0],
                     base_values=explainer.expected_value,
                     data=sample.values[0],
                     feature_names=X_test.columns.tolist())
)

In [ ]:
# ── 18. SAVE MODELS & ARTIFACTS ──────────────────────────────────────────────
import os
os.makedirs('/content/model_artifacts', exist_ok=True)

# Save models
joblib.dump(xgb_model,    '/content/model_artifacts/xgb_model.pkl')
joblib.dump(lgb_model,    '/content/model_artifacts/lgb_model.pkl')
joblib.dump(stack_model,  '/content/model_artifacts/stack_model.pkl')
joblib.dump(scaler,       '/content/model_artifacts/scaler.pkl')
joblib.dump(le_dict,      '/content/model_artifacts/label_encoders.pkl')

# Save config
config = {
    'optimal_threshold': float(optimal_threshold),
    'feature_cols': FEATURE_COLS,
    'model_auc': float(stack_auc),
    'pr_auc': float(average_precision_score(y_test, stack_proba)),
    'default_rate_train': float(y_train.mean()),
    'cost_fn': COST_FN,
    'cost_fp': COST_FP
}
with open('/content/model_artifacts/config.json', 'w') as f:
    json.dump(config, f, indent=2)

# SHAP values for dashboard
shap_df = pd.DataFrame({
    'feature': X_test.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False)
shap_df.to_csv('/content/model_artifacts/shap_importance.csv', index=False)

print('✅ All artifacts saved to /content/model_artifacts/')
!ls /content/model_artifacts/

In [ ]:
# ── 19. DOWNLOAD ALL ARTIFACTS ───────────────────────────────────────────────
!zip -r /content/loan_model_artifacts.zip /content/model_artifacts/
from google.colab import files
files.download('/content/loan_model_artifacts.zip')
print('Download started!')

## ✅ Next Steps
1. Download `loan_model_artifacts.zip`
2. Extract and place in your `webapp/` directory
3. Run `streamlit run app.py` (see `app.py` in the project repo)
4. Deploy on Render or Streamlit Cloud